# [LAB11] 지도학습 > 시계열 분석 > Prophet (하이퍼파라미터 튜닝)

## 📘 #01. 준비작업

### 📝 [1] 패키지 참조

### 📝 [2] 데이터셋 가져오기

In [ ]:
from hossam import *
from matplotlib import pyplot as plt
import seaborn as sb
import numpy as np
from pandas import to_datetime, DataFrame, DatetimeIndex, Series, date_range, concat
from prophet import Prophet
from prophet.plot import add_changepoints_to_plot
import holidays
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import ParameterGrid
from tqdm.auto import tqdm

In [ ]:
origin = load_data("covid19_seoul_230531")
origin.head()

### 📝 [3] 데이터 전처리

#### ✏ 필요한 데이터만 추출

#### ✏ 결측치 확인

#### ✏ 결측치 처리 (해당 날짜에 확진자가 발생하지 않은것으로 보고 0으로 채움)

In [ ]:
df = origin[['서울시 기준일', '서울시 추가 확진']].copy()
df.head()

In [ ]:
df.isna().sum()

#### ✏ 날짜 타입에 대한 형변환

### 📝 [4] 컬럼이름 변경, 정렬, 인덱스 재설정

### 📝 [5] 훈련, 검증 데이터 분리

In [ ]:
df2 = df.fillna(0)
df2.isna().sum()

In [ ]:
df2['서울시 기준일'] = to_datetime(df2['서울시 기준일'].str.strip(), format="%Y-%m-%d")
df2.dtypes

In [ ]:
df3 = df2.rename(columns={"서울시 기준일": "ds", "서울시 추가 확진": "y"})
df3.sort_values('ds', inplace=True)
df3.reset_index(drop=True, inplace=True)
df3.head()

In [ ]:
# 분할 비율
split_ratio = 0.8
split_idx = int(len(df3) * split_ratio)
train = df3.iloc[:split_idx]
test = df3.iloc[split_idx:]

### 📝 [7] 휴일 데이터 생성

#### ✏ 주말

#### ✏ 한국 기준 공휴일

In [ ]:
print("Train 기간:", train['ds'].min(), "~", train['ds'].max())
print("Valid 기간:", test['ds'].min(), "~", test['ds'].max())

In [ ]:
start_date = train['ds'].min()
end_date = test['ds'].max()
sat = date_range(start=start_date, end=end_date, freq='W-SAT')
sun = date_range(start=start_date, end=end_date, freq='W-SUN')
weekend = sat.union(sun)
df_weekend = DataFrame({"holiday": "weekend", "ds": weekend.sort_values(), "lower_window": 0, "upper_window": 0})
df_weekend.head()

In [ ]:
# 기간에 포함되는 연도 자동 추출
years = list(range(to_datetime(start_date).year, to_datetime(end_date).year + 1))
kr = holidays.KR(years=years)
kr

In [ ]:
holidays = {"holiday": [], "ds": [], "lower_window": [], "upper_window": []}
for date, name in kr.items():
    holidays["holiday"].append(name)
    holidays["ds"].append(date)
    holidays["lower_window"].append(0)
    holidays["upper_window"].append(0)
df_holidays = DataFrame(holidays)
df_holidays['ds'] = to_datetime(df_holidays['ds'])
df_holidays.sort_values('ds', inplace=True)
df_holidays.head()

### 📝 주말과 공휴일 병합

#### ✏ 학습 기간에 포함되는 데이터만 필터링

In [ ]:
holydays_final = concat([df_weekend, df_holidays], ignore_index=True)
holydays_final.sort_values('ds', inplace=True)
holydays_final.head(10)

In [ ]:
mask = (holydays_final['ds'] >= start_date) & (holydays_final['ds'] <= end_date)
holyday_final = holydays_final.loc[mask].reset_index(drop=True)
holyday_final.head()

## 📘 #02. Prophet 모델 구현

### 📝 [1] 튜닝하고자 하는 하이퍼파라미터 정의

### 📝 [2] GridSearchCV 구성

In [ ]:
params = ParameterGrid({
    'growth': ['linear'],
    'changepoint_prior_scale': [0.01, 0.1, 1.0],
    'seasonality_mode': ['additive', 'multiplicative'],
    'yearly_seasonality': [True],
    'weekly_seasonality': [True],
    'daily_seasonality': [False],
    'holidays': [holyday_final]
})
print('Total Possible Models', len(params))

In [ ]:
%%time
import logging
logging.getLogger("prophet").setLevel(logging.ERROR)
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
result = []
with tqdm(total=len(params)) as pbar:
    for i, p in enumerate(params):
        pbar.set_description(f"Model {i+1}/{len(params)}")
        m = Prophet(**p)
        m.fit(train)
        future = m.make_future_dataframe(periods=len(test), freq='D')
        forecast = m.predict(future)
        pred = forecast[['ds', 'yhat']][-len(test):]
        score = np.sqrt(mean_squared_error(test['y'].values, pred['yhat'].values))
        result.append({"score": score, "model": m, "params": p})
        pbar.update(1)
best_index = min(result, key=lambda x: x['score'])
best_model = best_index['model']
best_params = best_index['params']
best_score = best_index['score']
print("Best Score (RMSE):", best_score)
print("Best Parameters:", best_params)

## 📘 #03. 학습 결과 확인

### 📝 [1] 예측 데이터 생성

In [ ]:
# 실제 예측 데이터보다 7단계 더 미래까지 예측해보자.(1주일)
future = best_model.make_future_dataframe(periods=len(test)+7, freq='D')
forecast = best_model.predict(future)
forecast.head()

In [ ]:
fig = best_model.plot(forecast, figsize=(20, 7), xlabel='Date', ylabel='Passengers', uncertainty=True)
ax = fig.gca()
add_changepoints_to_plot(ax, best_model, forecast)
ax.set_title("시계열 예측")
sb.lineplot(data=test, x='ds', y='y', ax=ax, color='#ff6600', linestyle="--", label='test(real)')
plt.show()
plt.close()

In [ ]:
fig = best_model.plot_components(forecast, figsize=(20, 15))
ax = fig.gca()
plt.show()
plt.close()